# Q-Break — Track 5 results notebook

The hackathon working notebook. It runs the key symmetric experiments end to end through the
shared experiments runner (`qbreak.experiments.runner`), stores every run in the shared
results schema, and renders the charts: Grover iteration curve, scaling, noise sweeps,
quantum-vs-classical query comparison and quantum-counting accuracy.

All instances are **self-generated** (fixed messages, fixed seeds — see the README's Dataset
section). Set `QUICK = False` for the full suite (slower). Records go to a fresh directory so
the notebook is reproducible; set `RESULTS` to `None` to append to `backend/results` instead.

In [ ]:
import os, tempfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

QUICK = True
RESULTS = Path(tempfile.mkdtemp(prefix="qbreak-results-"))
if RESULTS is not None:
    os.environ["QBREAK_RESULTS_DIR"] = str(RESULTS)

import matplotlib.pyplot as plt
from qbreak.experiments import runner
from qbreak.experiments.schema import append_record, load_records, results_dir
from qbreak.experiments.aggregate import build_all

def run(name, **kwargs):
    records = runner.EXPERIMENTS[name](**({**runner.QUICK_ARGS[name], **kwargs} if QUICK else kwargs))
    for r in records:
        append_record(r)
    print(f"{name}: {len(records)} records")
    return records

print("results ->", results_dir())

## 1. Grover success vs iterations (4-bit, known beginning)
Measured probability on the true key against the theoretical sin²((2r+1)θ) envelope.

In [ ]:
run("grover_iteration_curve")
rows = build_all()["iteration_curve"]["rows"]
plt.plot([r["iterations"] for r in rows], [r["p_success"] for r in rows], "o-", label="measured (Aer)")
plt.plot([r["iterations"] for r in rows], [r["theory"] for r in rows], "--", label="theory sin²")
plt.xlabel("Grover iterations"); plt.ylabel("P(true key)"); plt.legend(); plt.title("Iteration curve"); plt.show()

## 2. Scaling: qubits and transpiled depth vs key size (4 → 12-bit)
Circuits are built and transpiled (not simulated), so 12-bit is cheap here. 16-bit is not simulated: ~30 qubits ≈ 16 GB of state vector.

In [ ]:
run("grover_scaling")
rows = build_all()["scaling"]["rows"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([r["size"] for r in rows], [r["qubits"] for r in rows], "o-"); ax[0].set(xlabel="key bits", ylabel="qubits")
ax[1].semilogy([r["size"] for r in rows], [r["transpiled_depth"] for r in rows], "o-"); ax[1].set(xlabel="key bits", ylabel="transpiled depth (Aer)")
plt.tight_layout(); plt.show()
for r in rows: print(r)

## 3. Noise: attack success vs depolarising error (4-bit)
Error p on every CX, p/10 on single-qubit gates; the circuit is transpiled to u + cx so every gate is noisy.

In [ ]:
run("grover_noise_sweep")
rows = [r for r in build_all()["noise"]["rows"] if r["model"] == "depolarizing"]
plt.semilogx([max(r["p"], 1e-4) for r in rows], [r["p_success"] for r in rows], "o-", label="P(true key)")
plt.axhline(rows[0]["baseline"], color="grey", ls=":", label="uniform guess 1/16")
plt.xlabel("depolarising p (CX)  [p=0 drawn at 1e-4]"); plt.ylabel("P(true key)"); plt.legend(); plt.title("Grover 4-bit under noise"); plt.show()
for r in rows: print(r["p"], round(r["p_success"], 3), "depth", r["transpiled_depth"])

### Fake IBM backend (FakeGuadalupeV2, 16 qubits)
The same attack transpiled to a real heavy-hex chip layout and run with its calibrated noise model.

In [ ]:
run("grover_fake_backend")
for r in build_all()["noise"]["rows"]:
    if r["model"] == "fake_backend":
        print(f"iterations={r['iterations']}  P(true key)={r['p_success']:.3f}  uniform={r['baseline']:.3f}  transpiled depth={r['transpiled_depth']:.0f}")

**Takeaway:** on today's noisy hardware even this miniature breach collapses towards random guessing — real breach testing needs fault tolerance.

## 4. Success rate per attack condition, and quantum vs classical queries

In [ ]:
run("grover_success_rate")
for r in build_all()["success_rate"]["rows"]:
    print(r["cipher"], r["size"], r["condition"], "success", r["success_rate"], "unique", r["unique_rate"], "runs", r["runs"])
rows = build_all()["comparison"]["rows"]
labels = [f"{r['size']}-bit {r['condition']}" for r in rows]
x = range(len(rows))
plt.bar([i - 0.2 for i in x], [r["quantum_oracle_calls"] for r in rows], 0.4, label="quantum oracle calls (counting + Grover)")
plt.bar([i + 0.2 for i in x], [r["classical_evaluations"] for r in rows], 0.4, label="classical cipher evaluations")
plt.xticks(list(x), labels, rotation=20); plt.ylabel("queries"); plt.legend(); plt.title("Query count per attack"); plt.show()

Query counts are the fair comparison; wall-clock is not (the simulator is far slower than brute force at these sizes). At 4-bit the quantum side also pays for quantum counting, so **no advantage is claimed**; the √N vs N/2 scaling is the point.

## 5. Quantum counting accuracy (estimated M vs true M)

In [ ]:
run("grover_counting_accuracy")
rows = build_all()["counting"]["rows"]
plt.scatter([r["true_m"] for r in rows], [r["estimated_m"] for r in rows], label="max-likelihood estimate")
plt.scatter([r["true_m"] for r in rows], [r["peak_m_estimate"] for r in rows], marker="x", label="single-peak reading")
top = max(r["true_m"] for r in rows) + 1
plt.plot([0, top], [0, top], ":", color="grey"); plt.xlabel("true M"); plt.ylabel("estimated M"); plt.legend(); plt.show()
print("exact:", sum(r["correct"] for r in rows), "/", len(rows))

## 6. What this means at real scale (published estimates)

In [ ]:
from qbreak.aes.resources import all_aes_estimates
est = all_aes_estimates()
for e in est["estimates"]:
    print(f"{e['cipher']}: {e['logical_qubits']['value']} logical qubits, T gates {e['t_gates']['text']}, "
          f"T-depth {e['t_depth']['text']}, Grover iterations {e['grover_iterations']['text']}  [GLRS16]")
print(); print(est["takeaway"]); print(); print(est["citations"]["GLRS16"])